# Data modeling 0: start here

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-00-start-here.ipynb)

Data modeling decides what a table means: what one row represents, how tables relate, and how history is
kept. A slow query can be tuned later. A table with the wrong grain reports the wrong revenue, however fast it
runs. This section teaches dimensional modeling for analytics in depth. It also covers the operational and
integration models that analytics data comes from, and the loading, testing and physical design that keep a
model correct as it grows.

Every lesson runs in DuckDB, an in-process analytical database, on the same small online shop. The SQL is
standard enough to carry over to Snowflake, BigQuery, Redshift, Databricks or Postgres; where a warehouse
differs, the lesson says so.

**Prerequisites:** SQL joins, `group by` and window functions ([SQL 2](../sql/sql-02-joins.ipynb), [SQL 5](../sql/sql-05-aggregations-and-ctes.ipynb) and [SQL 8](../sql/sql-08-window-functions.ipynb)).

*Last reviewed: November 2026, against DuckDB 2.0.*

## The lessons

| # | Lesson | You will be able to |
|---|---|---|
| 1 | [Grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) | Declare a grain, split facts from dimensions, and pick the right kind of fact table. |
| 2 | [Star schema design](data-modeling-02-star-schema-design.ipynb) | Design a star schema: surrogate keys, a date dimension, role-playing, junk and conformed dimensions. |
| 3 | [Slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb) | Keep dimension history with SCD types 1, 2, 3 and 6, and join facts to the version that was true. |
| 4 | [Relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb) | Find and fix double counting: fan-out, chasm traps, bridge tables and hierarchies. |
| 5 | [Normalized and operational models](data-modeling-05-normalized-and-operational-models.ipynb) | Normalize an operational schema, enforce it with constraints, and know when to denormalize. |
| 6 | [Warehouse architectures](data-modeling-06-warehouse-architectures.ipynb) | Compare Kimball, Inmon and Data Vault, and lay out raw, staging, core and mart layers. |
| 7 | [Loading and late data](data-modeling-07-loading-and-late-data.ipynb) | Load models incrementally and idempotently, and handle late facts and early facts. |
| 8 | [Physical design and performance](data-modeling-08-physical-design-and-performance.ipynb) | Shape tables for the engine: partitioning, sorting, file sizes, aggregates and wide tables. |
| 9 | [Testing, metrics and change](data-modeling-09-testing-metrics-and-change.ipynb) | Test models, define metrics once, and change a model without breaking its users. |
| 10 | [Interview case studies](data-modeling-10-interview-case-studies.ipynb) | Answer modeling interview questions by working through four case studies. |

Lessons 1 to 4 are the core of dimensional modeling; take them in order. Lessons 5 to 9 can be read in any order
after that. Lesson 10 brings everything together the way an interview does.

Practice is in the [Data modeling drills](data-modeling-drills.ipynb): exercises with expected output and hidden answers. The [Data modeling reference](data-modeling-reference.ipynb) lists every
pattern on one page.

## How each lesson works

Each `##` section explains one idea: how it works, where it matters, and what to watch out for. Then a runnable
example builds a table or runs a query, and a **What to notice** note points at the row, count or column in the
output that proves the point. Read the output before the note. Each lesson ends with questions to check yourself
and a short list of takeaways.

Run the cells in order. Every lesson starts from the same source tables, so each one runs on its own; nothing
carries over between notebooks.

## The shop and its source data

The shop sells coffee, tea, mugs and kettles online and in one store. Its order application and CRM give us seven
extracts. They are the **sources**: shaped for running the shop, not for answering questions about it.

| Table | One row per | Notes |
|---|---|---|
| `src_customers` | customer | the CRM's *current* city and segment only |
| `src_customer_changes` | change to a customer | the CRM audit log: customer 102 became `enterprise` on Feb 1, customer 104 moved from Hull to Leeds on Feb 15 |
| `src_products` | product (SKU) | category and department are text on each product |
| `src_orders` | order | channel, status, a shipping fee, and shipped and delivered timestamps |
| `src_order_lines` | product on an order | quantity, unit price and a line discount |
| `src_payments` | payment | order 1003 was paid in two parts; 1008 is unpaid; 1009 was cancelled |
| `src_promotions`, `src_order_promotions` | promotion; promotion on an order | order 1003 used two promotions |

```text
src_customers 1───* src_orders 1───* src_order_lines *───1 src_products
      │                 │  1
      │                 ├───* src_payments
      1                 └───* src_order_promotions *───1 src_promotions
      *
src_customer_changes
```

Over the lessons these become a star schema: `fact_order_line` and `fact_payment` surrounded by `dim_date`,
`dim_customer`, `dim_product` and others.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_customer_changes` (the CRM's audit log of segment and city changes), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order), `src_payments` (one row per payment; an order can have several) and `src_promotions` and `src_order_promotions` (promotions applied to orders): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_customer_changes (customer_id int, changed_at timestamp, city varchar, segment varchar);
insert into src_customer_changes values  -- the CRM's audit log: one row each time a customer's details changed
    (101, '2025-11-02 00:00', 'Leeds', 'smb'),
    (102, '2025-11-20 00:00', 'York',  'smb'),
    (102, '2026-02-01 00:00', 'York',  'enterprise'),
    (103, '2025-12-05 00:00', 'Bath',  'enterprise'),
    (104, '2026-01-08 00:00', 'Hull',  'smb'),
    (104, '2026-02-15 00:00', 'Leeds', 'smb'),
    (105, '2026-01-19 00:00', 'Derby', 'smb'),
    (106, '2026-02-10 00:00', 'York',  'mid-market');
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
create table src_promotions (promo_code varchar primary key, promo_name varchar);
insert into src_promotions values ('WINTER10', 'Winter sale'), ('BULK', 'Bulk discount'), ('LOYAL', 'Loyalty reward');
create table src_order_promotions (order_id int, promo_code varchar, primary key (order_id, promo_code));
insert into src_order_promotions values (1002, 'WINTER10'), (1003, 'BULK'), (1003, 'LOYAL'), (1006, 'BULK');
""")
con.sql("show tables").show()

## A first look

Here is one order, 1003, across the source tables. Reading it shows the questions a model has to settle.

In [ ]:
con.sql("""
select o.order_id, o.order_ts, c.name, c.segment, l.line_no, p.product_name, l.qty, l.unit_price, l.discount
from src_orders o
join src_customers c using (customer_id)
join src_order_lines l using (order_id)
join src_products p using (sku)
where o.order_id = 1003
order by l.line_no
""").show()

Order 1003 has two lines, two payments and two promotions. Join all of them to the order in one query:

In [ ]:
con.sql("""
select l.line_no, l.sku, l.qty * l.unit_price - l.discount as line_amount, p.payment_id, p.amount as paid, op.promo_code
from src_order_lines l
join src_payments p using (order_id)
join src_order_promotions op using (order_id)
where l.order_id = 1003
order by all
""").show()

In [ ]:
con.sql("""
select (select sum(qty * unit_price - discount) from src_order_lines where order_id = 1003) as true_sales,
       sum(l.qty * l.unit_price - l.discount)                                           as sales_after_join
from src_order_lines l
join src_payments p using (order_id)
join src_order_promotions op using (order_id)
where l.order_id = 1003
""").show()

**What to notice:** Each of the 2 lines repeats once for every payment and every promotion: 2 × 2 × 2 = 8 rows, and sales of 232.50
become 930.00, four times too much. Nothing in the query is wrong SQL; the tables simply have different
*grains* (one row per line, per payment, per promotion). The first query hides a second problem: `segment` comes
from the CRM's current row. Customer 103 never changed, but customer 102 became `enterprise` on February 1, so
the same join would label 102's January order `enterprise` too. Grain is lesson 1, joining tables of different
grains is lesson 4, and keeping history is lesson 3.

## Where to go next

Start with [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb). If you are preparing for an interview, read [Data modeling 10: interview case studies](data-modeling-10-interview-case-studies.ipynb) first to see what is asked, then come
back to lesson 1.

Related material elsewhere in the handbook: [SQL 11: data modeling and query optimization](../sql/sql-11-modeling-and-optimization.ipynb)
is a shorter SQL-first introduction, [dbt](../dbt/dbt.ipynb) shows how models are built and tested in a project,
and the [semantic layer](../concepts/semantic-layer.ipynb) builds on lesson 9's metric definitions.